# Random Forests

**Notebook 3 of 5.** Bagging works because the trees make different mistakes. They still
make too many of the same ones, and the fix is a second piece of randomness — the one that
turns bagging into a **random forest**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import root_mean_squared_error

## 1. The data

`data/concrete.csv` again, same split and `random_state=42`, so these scores sit beside
notebook 02's **5.52 MPa** for bagging.

In [ ]:
df = pd.read_csv("../data/concrete.csv")

X = df.drop(columns="compressive_strength")
y = df["compressive_strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training mixes, {len(X_test)} test mixes, {X.shape[1]} features")

## 2. The trees still agree too much

`RandomForestRegressor` with `max_features=1.0` lets every split consider all eight features,
which is exactly the bagging of notebook 02. Ask those trees what question they open with.

In [ ]:
bagged = RandomForestRegressor(
    n_estimators=200, max_features=1.0, random_state=42, n_jobs=-1
)
bagged.fit(X_train, y_train)

# tree_.feature[0] is the feature the root node splits on
roots = pd.Series([X.columns[t.tree_.feature[0]] for t in bagged.estimators_])
print(roots.value_counts().to_string())

**182 of the 200 open with `age`**, the other 18 with `cement`, and six of the eight
ingredients never appear at the root.

That is the limit of bootstrapping alone. Resampling changes which mixes a tree sees, but
`age` is the strongest predictor in almost every resample, so the trees come out shaped
alike — and averaging only cancels errors that differ.

## 3. The second randomness

A random forest adds one rule to bagging:

> At **every split**, choose the best question from a **random subset of the features**
> instead of from all of them.

`max_features` sets the size of that subset. A tree denied `age` at some node must ask
something else, so the forest fills with trees shaped differently on purpose.

The cost is that each tree is **worse**, sometimes forbidden the best question. The next
section measures both halves of that trade.

This helper measures the second and third of those:

> **Exercise:**
>
> For `max_features` of `1.0`, `0.5` and `"sqrt"`, fit a 200-tree forest and print three
> things: the RMSE of the forest, the mean RMSE of its individual trees, and how correlated
> the trees' test predictions are with each other.
>
> `tree_agreement` above gives you the last two.

<details><summary>
Click here for a hint...
</summary>

`forest.estimators_` is the list of fitted trees. Each one predicts from a plain array, so
pass `X_test.to_numpy()`.

</details>

In [ ]:
def tree_agreement(forest):
    """Measure how alike a forest's trees are on the test set.

    Args:
        forest: a fitted forest, whose `estimators_` are the individual trees.

    Returns:
        tuple: (mean RMSE of a single tree, mean correlation between two trees).
    """
    # one row of test predictions per tree
    predictions = np.array([t.predict(X_test.to_numpy()) for t in forest.estimators_])
    rmses = [root_mean_squared_error(y_test, row) for row in predictions]

    # the upper triangle holds each pair of trees once, without the diagonal of 1s
    correlations = np.corrcoef(predictions)
    pairs = np.triu_indices_from(correlations, k=1)

    return np.mean(rmses), correlations[pairs].mean()

In [ ]:
print(f"{'max_features':>14}  {'forest':>7}  {'one tree':>9}  {'correlation':>12}")
for setting in [1.0, 0.5, "sqrt"]:
    forest = RandomForestRegressor(
        n_estimators=200, max_features=setting, random_state=42, n_jobs=-1
    )
    forest.fit(X_train, y_train)
    alone, agreement = tree_agreement(forest)
    print(f"{str(setting):>14}  "
          f"{root_mean_squared_error(y_test, forest.predict(X_test)):7.2f}  "
          f"{alone:9.2f}  {agreement:12.3f}")

<details><summary>
Click here to compare your answer...
</summary>

| `max_features` | forest | one tree | correlation |
| :-- | --: | --: | --: |
| 1.0, all eight | 5.54 | 7.93 | 0.865 |
| 0.5, four | **5.33** | 8.36 | 0.827 |
| `"sqrt"`, three | 5.51 | 9.48 | 0.749 |

Both columns move at once. Eight features to four makes **each tree worse**, 7.93 to 8.36,
and **the forest better**, 5.54 to 5.33, because the trees agree less.

Three features keeps the first half and loses the second: the trees drop to 9.48 and the
forest worsens too. There is a floor, and on eight features it is around four.

</details>

## 4. A default worth knowing

The randomness of section 3 is **off by default for regression**:

| | default `max_features` | which means |
| :-- | :-- | :-- |
| `RandomForestRegressor` | `1.0` | every feature at every split — bagging |
| `RandomForestClassifier` | `"sqrt"` | three of eight here |

So a regression forest out of the box is a bagged forest, and its 5.54 beside notebook 02's
5.52 is the same method twice. **For regression you have to ask for the randomness**, with
`max_features` below 1.0. For classification it is already on.

## 5. One more step: Extra Trees

A forest picks the **best** threshold for each feature it may consider. **Extremely
randomized trees** pick a **random** threshold for each, and keep the best of those.

In [ ]:
extra = ExtraTreesRegressor(
    n_estimators=200, max_features=0.5, random_state=42, n_jobs=-1
)
extra.fit(X_train, y_train)

alone, agreement = tree_agreement(extra)
print(f"extra trees: {root_mean_squared_error(y_test, extra.predict(X_test)):.2f} MPa, "
      f"one tree {alone:.2f}, correlation {agreement:.3f}")

**5.06 MPa**, the best here, from trees that each score 7.89. They are also faster: a forest
tries every threshold of every feature it considers and extra trees try one, so this fit took
0.38 seconds against the forest's 0.93.

Random thresholds are not always better. They add variance to each tree, which helps while
the trees are too alike and hurts once they are not. Both are worth trying.

## Summary

- Bagged trees stay **shaped alike**: 182 of 200 opened with `age`.
- A random forest adds a **random subset of features at every split**.
- The trade runs both ways: fewer features make **each tree worse** and **the forest better**,
  until too few makes everything worse. Four of eight was best, at 5.33.
- **`RandomForestRegressor` defaults to all features**, which is bagging; the classifier does
  not.
- **Extra trees** randomize the thresholds too: 5.06 MPa, and faster.

Every score so far came from the test set, which is not what a test set is for. Next, in
`04_out_of_bag.ipynb`: the rows each tree never saw.